In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from datetime import datetime

In [ ]:
BATCH_SIZE = 16
data_dir = 'synthetic_dataset.json'

In [ ]:
with open(data_dir, 'r', encoding='utf-8') as f:
    data = json.load(f)

In [ ]:
vocab = {"<PAD>": 0, "<UNK>": 1}
tags = {"O": 0, "B-ITEM": 1, "B-PRICE": 2, "B-PERSON": 3, "B-MULTIPLIER": 4}

In [ ]:
X, Y_ner, Y_reg = [], [], []
max_len = 30

In [ ]:
for item in data:
    text = item['raw_text'].lower().split()
    entities = item['entities']
        
    # Simple Regression Target: Normalized count of entities (0.0 to 1.0)
    complexity_score = min(len(entities) / 10.0, 1.0) 
        
    seq_x, seq_y = [], []
    for word in text[:max_len]:
        if word not in vocab: vocab[word] = len(vocab)
        seq_x.append(vocab[word])
        
        # Simple Substring matching for Baseline
        assigned_tag = tags["O"]
        for ent in entities:
            if ent['value'].lower() in word:
                assigned_tag = tags[f"B-{ent['entity']}"]
                break
        seq_y.append(assigned_tag)
            
    # Pad sequences
    seq_x += [vocab["<PAD>"]] * (max_len - len(seq_x))
    seq_y += [tags["O"]] * (max_len - len(seq_y))
        
    X.append(seq_x)
    Y_ner.append(seq_y)
    Y_reg.append(complexity_score)

In [ ]:
X = np.array(X) 
Y_ner = np.array(Y_ner) 
Y_reg = np.array(Y_reg, dtype=np.float32)

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices((X, Y_ner, Y_reg)).batch(BATCH_SIZE)

In [ ]:
@tf.keras.utils.register_keras_serializable()
class TemporalAttention(layers.Layer):
    """Custom Layer Requirement: Focuses on important tokens."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], 1), initializer="random_normal", trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        score = tf.matmul(inputs, self.W)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * inputs
        return tf.reduce_sum(context_vector, axis=1)

In [ ]:
def custom_combined_loss(y_true_ner, y_pred_ner, y_true_reg, y_pred_reg):
    """Custom Loss Function Requirement: Combines CrossEntropy and Huber Loss."""
    scce = tf.keras.losses.SparseCategoricalCrossentropy()(y_true_ner, y_pred_ner)
    huber = tf.keras.losses.Huber()(y_true_reg, y_pred_reg)
    return scce + (0.5 * huber) # Weighing NER slightly higher

In [ ]:
vocab_size = len(vocab)
num_tags = len(tags)
max_len = 30

In [ ]:
inputs = layers.Input(shape=(max_len,), dtype=tf.int32, name="input_ids")
embedding = layers.Embedding(input_dim=vocab_size, output_dim=64)(inputs)

In [ ]:
lstm_out = layers.Bidirectional(layers.LSTM(32, return_sequences=True))(embedding)

In [ ]:
ner_output = layers.Dense(num_tags, activation='softmax', name='ner_head')(lstm_out)

In [ ]:
context_vector = TemporalAttention()(lstm_out)
reg_output = layers.Dense(1, activation='linear', name='reg_head')(context_vector)

In [ ]:
model = Model(inputs=inputs, outputs=[ner_output, reg_output])

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

In [ ]:
train_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()
train_mae_metric = tf.keras.metrics.MeanAbsoluteError()

In [ ]:
logdir = "logs/train/" + datetime.now().strftime("%Y%m%d-%H%M%S")
summary_writer = tf.summary.create_file_writer(logdir)

In [ ]:
@tf.function
def train_step(x, y_ner, y_reg):
    with tf.GradientTape() as tape:
        pred_ner, pred_reg = model(x, training=True)
        # Custom Loss
        loss = custom_combined_loss(y_ner, pred_ner, y_reg, pred_reg)

    gradients = tape.gradient(loss, model.trainable_weights)
    optimizer.apply_gradients(zip(gradients, model.trainable_weights))
    
    # Update metrics
    train_acc_metric.update_state(y_ner, pred_ner)
    train_mae_metric.update_state(y_reg, pred_reg)
    return loss

In [ ]:
EPOCHS = 30
for epoch in range(EPOCHS):
    for step, (x_batch, y_ner_batch, y_reg_batch) in enumerate(dataset):
        loss_value = train_step(x_batch, y_ner_batch, y_reg_batch)
        
    acc = train_acc_metric.result()
    mae = train_mae_metric.result()
    
    # Write to TensorBoard
    with summary_writer.as_default():
        tf.summary.scalar('Loss', loss_value, step=epoch)
        tf.summary.scalar('Accuracy', acc, step=epoch)
        tf.summary.scalar('MAE', mae, step=epoch)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss_value:.4f} | Acc: {acc:.4f} | MAE: {mae:.4f}")
    
    train_acc_metric.reset_states()
    train_mae_metric.reset_states()

In [ ]:
model.save("talangin_model.keras")